# 1. Import libraries and data

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# from wifiplotting import OSMPlotContext, plot_wifi_heatmap, TL_CORNER, BR_CORNER
import wifiplotting as wp
from wifiplotting import TL_CORNER, BR_CORNER

In [ ]:
wifi_df = pd.read_csv('../data/sequoia.csv')

In [ ]:
wifi_df.head()

In [ ]:
osm_context = wp.OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]])
fig, ax, osm_metadata = osm_context.generate_base_axis()
osm_metadata

Main idea: we know that we should model location and indoor/outdoor, but should we model time? We check by doing simple KNN for location and indoor/outdoor, then seeing if the residuals have any relation to time of day.

Potential problem: because of sampling design, it is possible location and time are correlated.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, Normalizer

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))

In [ ]:
wifi_df.head(5)

In [ ]:
numeric_to_scale = ["longitude", "latitude"]   # columns that need normalization

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", scaler, numeric_to_scale),
    ]
)

pipeline = Pipeline([
    # ("preprocess", preprocessor),
    ("model", KNeighborsRegressor())
])

param_grid = {
    "model__n_neighbors": np.arange(30, 71),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]  # Manhattan vs Euclidean
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",  # or "r2"
    n_jobs=-1,   # use all CPU cores
    verbose=0
)

In [ ]:
var = 'rssi_sample' # 'noise'

# df_knn = wifi_df[wifi_df[var].notna()].copy()
df_knn = wifi_df[wifi_df[var].notna()].copy()
# aggregate duplicate X values
# note: this may no longer be necessary after aggregating in preprocessing
# df_knn = df_knn.groupby(["latitude", "longitude", "indoor"], as_index=False)[var].mean()
grid.fit(df_knn[["longitude", "latitude", "indoor"]], df_knn[var])

In [ ]:
print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

In [ ]:
grid_width = 100

x_new = np.linspace(0, 1, grid_width)
y_new = np.linspace(0, 1, grid_width)
grid_points = np.stack(np.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)
coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T

osm_context = wp.OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]])

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)

X_new = np.concatenate([coord_grid, z_new], axis=-1)

In [ ]:
grid.best_params_

In [ ]:
new_data = pd.DataFrame(X_new, columns=['longitude', 'latitude', 'indoor'])

In [ ]:
y_hat = grid.best_estimator_.predict(new_data)

In [ ]:
base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(figsize=(6,6))
wlon, wlat = osm_context.to_world(long_new, lat_new)
base_ax.scatter(wlon, wlat, c=y_hat, cmap="RdYlGn")
wlon, wlat = osm_context.to_world(wifi_df.longitude, wifi_df.latitude)
sc = base_ax.scatter(wlon, wlat, c=wifi_df.rssi_sample, cmap="RdYlGn")
cbar = base_fig.colorbar(sc, ax=base_ax)
base_fig.tight_layout()
plt.savefig("../writeups/milestone3/knnplot.pdf", dpi = 600)

In [ ]:
train_err = df_knn['rssi_sample'] - grid.best_estimator_.predict(df_knn[['longitude', 'latitude', 'indoor']])

In [ ]:
np.sqrt((train_err**2).mean())

<div style="display:none">
$
\newcommand{\rssi}{\texttt{RSSI}}
\newcommand{\GP}{\mathcal{GP}}
\newcommand{\K}{\mathcal{K}}
\newcommand{\R}{\mathbb{R}}
\newcommand{\N}{\mathcal{N}}
$
</div>

## Setup

Define the variables of interest.
$$r = \rssi, \quad x = \texttt{longitude}, \quad y = \texttt{latitude}, \quad z = \texttt{indoor}$$

Assume that the signal field is described by a latent function $f$.
$$r_i = f(x_i, y_i, z_i) + \epsilon_i$$
where $\epsilon_i \sim \N(0, \sigma^2_i)$. How should we estimate the noise variance? Do we want to also model variance as a function of $x_i, y_i, z_i$?

<br><br>

## Model

We put a Gaussian process (GP) prior on $f$.
$$f \sim \GP(m, \K)$$
where $m: \R^3 \to \R$ mean function, $\K: \R^3 \times \R^3 \to \R$ covariance kernel. How to set $m$ and $\K$? We see that changes in $z$ lead to much sharper changes in $r$ compared to changes in $x$ and $y$. Can we capture this in our kernel?

<br>

Then, we have the full model below.
$$f \sim \GP(m, \K)$$
$$\vec{r} \mid f \sim \N(f(\vec{x}, \vec{y}, \vec{z}), \Sigma_0)$$
$$\Sigma_0 = \text{noise covariance matrix } \textrm{diag}(\sigma^2_i)$$
Note that under independent Gaussian noise with known/fixed variance, the posterior for this model is available in closed form. If we have unknown variance but homoskedasticity, a simple MCMC solution is available.

<br><br>

## More Advanced Directions

- Our measurements for $x_i$ and $y_i$ are noisy; how can we account for this in our model?
- A refined approach is to account for the WiFi access points on campus, for which we also have data. A reasonable assumption is that there is an access point in each building, though we could treat their locations as unknown parameters as well. Then, there are multiple ways to incoporate this into our model.

  The following are possible ammendments to the model.

  Global GP:
  - Let $d_i$ be the distance of point $(x_i, y_i)$ to its access point. Account for $d_i$ in $m$ and $\K$.
  - Account for access point $a_j$ in $m$ and $\K$. For new observations, set $\hat{f}(x_{new}, y_{new}, z_{new}) = \max_j f(x_{new}, y_{new}, z_{new}, a_j)$. Can also model access point as a random parameter, inducing a hierarchical structure.
  - Condition on additional constraints on the GP dynamics to reflect behavior (e.g. point repellers).

  <br>
  
  Local GPs: For each access point $a_j$, set a different GP prior $f_j \sim \GP(m, \K)$.
  - Define $\displaystyle f(x_i, y_i, z_i) = \sum_j \mathbb{1}\{j = \arg\min_j d(i, j)\} f_j(x_i, y_i, z_i)$, or $\displaystyle f(x_i, y_i, z_i) = \max_j f_j(x_i, y_i, z_i)$

 
    Hard assignments are problematic due to discontinuities on boundaries. Also, location and inferred strength don't necessarily directly correspond with access point due to potential noise and the tendency of devices to "stick" with suboptimal access points until connection becomes too bad.
  
  - Define $\displaystyle f(x_i, y_i, z_i) = \sum_j w_j(x_i, y_i) f_j(x_i, y_i, z_i)$

 
    where $w_j$ is a weight function for access point $j$ based on distance. A mixture reflects the "stickiness."

In [ ]:
wifi_df['ap'] = wifi_df['ap'].astype('category').cat.codes
fig, ax, noise_points, na_points = wp.plot_agg_wifi_heatmap(wifi_df, value="ap", plotname="Access Points", cmap=plt.colormaps["turbo"], context=osm_context)
rows_used = int(noise_points["sample_count"].sum())
print(f"Noise heatmap uses {len(noise_points)} unique coordinate bins from {rows_used} rows.")
plt.show()